# 20 Newsgroups Topic Classification

**Dataset:** the classic 20 Newsgroups corpus — ~18,800 Usenet posts across 20 topics (space, hockey, cryptography, religion, guns, motorcycles…). **Task:** assign each post to its newsgroup from text alone. The fixed train/test split is chronological, so the model is tested on later posts.

In [1]:
from sklearn.datasets import fetch_20newsgroups
tr = fetch_20newsgroups(subset='train', random_state=42)
te = fetch_20newsgroups(subset='test', random_state=42)
print('Train:', len(tr.data), 'Test:', len(te.data))
print(tr.target_names)

Train: 11314 Test: 7532
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

pipe = make_pipeline(TfidfVectorizer(max_features=30000, ngram_range=(1, 2), sublinear_tf=True),
                     LogisticRegression(max_iter=1000))
pipe.fit(tr.data, tr.target)
pred = pipe.predict(te.data)
print('Accuracy:', round(accuracy_score(te.target, pred), 4))
print(classification_report(te.target, pred, target_names=te.target_names))

Accuracy: 0.8259
                          precision    recall  f1-score   support

             alt.atheism       0.77      0.73      0.75       319
           comp.graphics       0.65      0.77      0.70       389
 comp.os.ms-windows.misc       0.76      0.74      0.75       394
comp.sys.ibm.pc.hardware       0.70      0.72      0.71       392
   comp.sys.mac.hardware       0.82      0.81      0.82       385
          comp.windows.x       0.84      0.74      0.79       395
            misc.forsale       0.83      0.91      0.87       390
               rec.autos       0.90      0.89      0.90       396
         rec.motorcycles       0.93      0.94      0.94       398
      rec.sport.baseball       0.88      0.93      0.90       397
        rec.sport.hockey       0.94      0.96      0.95       399
               sci.crypt       0.96      0.89      0.93       396
         sci.electronics       0.71      0.77      0.74       393
                 sci.med       0.87      0.82      0.84   

In [3]:
samples = [
 'The shuttle launch was delayed again because of a faulty oxygen sensor on the pad.',
 'The goalie made an incredible save in overtime and the crowd went wild.',
 'You should encrypt your email with a strong cipher if you care about privacy.',
 'The new graphics card benchmarks are twice as fast as last year model.',
]
for s in samples:
    print(tr.target_names[pipe.predict([s])[0]], '<-', s[:60] + '...')

sci.space <- The shuttle launch was delayed again because of a faulty oxy...
rec.sport.hockey <- The goalie made an incredible save in overtime and the crowd...
sci.crypt <- You should encrypt your email with a strong cipher if you ca...
comp.graphics <- The new graphics card benchmarks are twice as fast as last y...


## What I learned
- Word 2-grams matter: 'ice hockey' and 'space shuttle' are single ideas split across tokens.
- The residual confusion is genuinely hard pairs (talk.politics vs talk.religion, comp.sys.mac vs comp.windows) where the topics themselves overlap.
- Headers/footers leak group identity (signatures, quoted paths) — the strict version strips them and scores lower; this run keeps the standard fetch defaults so results compare with the literature.